In [1]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle

In [9]:
df_cell_median = pd.read_pickle("out/animation/df_cell_median.pkl")
df_agents = pd.read_pickle("out/animation/df_agents.pkl")

In [14]:


# ================= settings =================
N_SIDE = 7
NODE_SIZE = 900      # max circle area (points^2)
ARRIVAL_GAP = 5      # space between arrow tip and destination circle (points)
CURVE = 0.25         # arrow curvature (0 = straight)
MARGIN = 0.1         # space around each grid so curved arrows are never cut (in cells)

WINDOW = 5           # flows: sum moves over the last WINDOW steps
MIN_FLOW = 1         # default minimum, used when a class doesn't set its own
FPS = 2
HOLD_LAST = 8        # extra frames on the last step (pause before looping)

CELL_COLOR = "white"       # flow panels: cell fill
BORDER_COLOR = "#363636"   # flow panels: cell separators
BORDER_WIDTH = 1.5
BG_COLOR = "white"         # whole figure

CENTROID_SIZE = 0    # black dot at each cell centre (points^2); 0 = no centroids

# ---- layout (inches) ----
PANEL = 4.7          # side of EVERY panel (all four grids get exactly the same size)
COL_GAP = 0.5        # horizontal space between panels
SIDE = 0.3           # left/right figure margin
LEGEND_H = 0.8       # height reserved for the legend at the bottom
TITLE_H = 0.6        # space for the titles above the bottom panels
TOP_H = 0.45         # space above the top row (richness title)

FIG_W = 2 * SIDE + 3 * PANEL + 2 * COL_GAP
COL_X = [SIDE + PANEL / 2 + k * (PANEL + COL_GAP) for k in range(3)]   # column centres
ROW_Y = [LEGEND_H + 0.1 + PANEL + TITLE_H, LEGEND_H + 0.1]           # bottom edge of top / bottom row
FIG_H = ROW_Y[0] + PANEL + TOP_H

CBAR_FRAC = 0.6      # colorbar height as a fraction of the grid height
CBAR_W = 0.25        # colorbar width
CBAR_GAP = 0.25      # space between grid and colorbar
CBAR_LABEL_SIZE = 16

LEGEND_COLOR = "#8a8a8a"      # legend shapes
LEGEND_TEXT_COLOR = "#555555" # legend text
LEGEND_FONTSIZE = 14
STEP_FONTSIZE = 13

# ---- step slider ----
SLIDER_W = 2.2       # slider length (inches)
SLIDER_H = 0.6       # slider area height (inches)
SLIDER_TOP = 0.3     # distance from the top of the figure (inches)
SLIDER_LW = 4        # track thickness (points)
SLIDER_KNOB = 80     # knob size (points^2)
SLIDER_COLOR = "#555555"
SLIDER_TRACK = "#dddddd"
SLIDER_LABEL_SIZE = 12

CLASSES = {
    "A": dict(color="#2a4b7a", name="Reddito alto",  min_flow=1),
    "B": dict(color="#00957c", name="Reddito medio", min_flow=6),
    "C": dict(color="#f54333", name="Reddito basso", min_flow=6),
}

# ================= data prep: richness =================
grids = (df_cell_median.reindex(columns=range(N_SIDE**2))
         .to_numpy().reshape(-1, N_SIDE, N_SIDE))
grid_row = {step: k for k, step in enumerate(df_cell_median.index)}   # step -> frame of `grids`

# ================= data prep: flows =================
def as_tuple(v):
    # works whether positions are real tuples or strings like "(3, 5)" read back from CSV
    return ast.literal_eval(v) if isinstance(v, str) else tuple(v)

ag = df_agents.reset_index()
for c in ["pos", "new_position", "source"]:
    ag[c] = ag[c].map(as_tuple)
ag["moved"] = (ag["new_position"].map(lambda p: p != (-1, -1))
               & (ag["source"] != ag["pos"]))

# Only animate the steps present in BOTH datasets, so the panels stay in sync
steps = np.array(sorted(set(ag["Step"].unique()) & set(df_cell_median.index)))

def flows_at(flows, step, min_flow=MIN_FLOW):
    f = flows[(flows["Step"] > step - WINDOW) & (flows["Step"] <= step)]
    f = f.groupby(["source", "pos"])["n"].sum()
    return f[f >= min_flow]

flow_data = {}
for tipo in CLASSES:
    sub = ag[ag["tipo"] == tipo]
    mf_min = CLASSES[tipo].get("min_flow", MIN_FLOW)
    pop = sub.groupby(["Step", "pos"]).size()
    flows = (sub[sub["moved"]].groupby(["Step", "source", "pos"])
             .size().rename("n").reset_index())
    max_flow = max((flows_at(flows, s, mf_min).max() if len(flows_at(flows, s, mf_min)) else 0)
                   for s in steps) or 1
    flow_data[tipo] = dict(pop=pop, flows=flows, max_pop=pop.max(),
                           max_flow=max_flow, min_flow=mf_min)

def xy(cell):
    """Grid cell (x, y) -> plot coords, same layout as the heatmap (row = x, column = y)."""
    x, y = cell
    return y + 0.5, x + 0.5

def curve_sign(p0, p1, reverse_exists):
    """Bend the arrow inward if it would bulge outside the grid,
    unless the opposite arrow exists (then keep the default so the two don't overlap)."""
    (x0, y0), (x1, y1) = p0, p1
    px = (x0 + x1) / 2 - 0.5 * CURVE * (y1 - y0)
    py = (y0 + y1) / 2 + 0.5 * CURVE * (x1 - x0)
    outside = not (0 <= px <= N_SIDE and 0 <= py <= N_SIDE)
    return -1 if (outside and not reverse_exists) else 1

def draw_flows(ax, tipo, step):
    d = flow_data[tipo]
    color, name = CLASSES[tipo]["color"], CLASSES[tipo]["name"]
    ax.clear()
    ax.set_facecolor(BG_COLOR)
    ax.set_xlim(-MARGIN, N_SIDE + MARGIN)
    ax.set_ylim(N_SIDE + MARGIN, -MARGIN)            # row 0 on top
    ax.set_aspect("equal"); ax.axis("off")

    for r in range(N_SIDE):
        for c in range(N_SIDE):
            ax.add_patch(Rectangle((c, r), 1, 1, facecolor=CELL_COLOR,
                                   edgecolor=BORDER_COLOR, linewidth=BORDER_WIDTH, zorder=1))

    pop = d["pop"]
    p = pop.loc[step] if step in pop.index.get_level_values(0) else pd.Series(dtype=float)
    radius = {}
    for cell, n in p.items():
        size = NODE_SIZE * n / d["max_pop"]
        radius[cell] = np.sqrt(size / np.pi)
        ax.scatter(*xy(cell), s=size, color=color, alpha=0.35, edgecolor="none", zorder=2)

    if CENTROID_SIZE > 0:
        cx, cy = np.meshgrid(np.arange(N_SIDE) + 0.5, np.arange(N_SIDE) + 0.5)
        ax.scatter(cx.ravel(), cy.ravel(), s=CENTROID_SIZE, color="black", zorder=4)

    f = flows_at(d["flows"], step, d["min_flow"])
    edges = set(f.index)
    mf = d["max_flow"]
    for (src, dst), n in f.items():
        p0, p1 = xy(src), xy(dst)
        rad = CURVE * curve_sign(p0, p1, (dst, src) in edges)
        ax.add_patch(FancyArrowPatch(
            p0, p1, connectionstyle=f"arc3,rad={rad}",
            arrowstyle="-|>", mutation_scale=6 + 10 * n / mf,
            linewidth=0.3 + 6 * n / mf, color=color, alpha=0.35 + 0.6 * n / mf,
            shrinkA=3, shrinkB=radius.get(dst, 0) + ARRIVAL_GAP,
            clip_on=False, zorder=3))

    n_moves = int(f.sum()) if len(f) else 0
    ax.set_title(f"{name}\n{n_moves} spostamenti", fontsize=LEGEND_FONTSIZE+2, color=color,
                 weight="bold", pad=2, linespacing=1.4)

# ================= figure =================
def add_panel(fig, cx, y0, size=PANEL):
    """Square axes of `size` inches, centred horizontally on cx, bottom edge at y0 (inches)."""
    return fig.add_axes([(cx - size / 2) / FIG_W, y0 / FIG_H, size / FIG_W, size / FIG_H])

with plt.style.context("default"):
    plt.rcParams.update({"font.size": 13})
    fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor=BG_COLOR)

    ax_left  = add_panel(fig, COL_X[0], ROW_Y[0]); ax_left.axis("off")    # free space for your text
    ax_rich  = add_panel(fig, COL_X[1], ROW_Y[0])
    ax_right = add_panel(fig, COL_X[2], ROW_Y[0]); ax_right.axis("off")   # free space for your text
    ax_flows = {t: add_panel(fig, COL_X[k], ROW_Y[1]) for k, t in enumerate(CLASSES)}

    # --- richness panel: same limits as the flow panels -> identical grid size ---
    cmap = plt.get_cmap("Purples").copy()      # light = poorer, dark = richer
    cmap.set_bad("#d9d9d9")
    norm = LogNorm(vmin=np.nanmin(grids), vmax=np.nanmax(grids))
    mesh = ax_rich.pcolormesh(np.ma.masked_invalid(grids[grid_row[steps[0]]]),
                              cmap=cmap, norm=norm, edgecolors="white", linewidth=3)
    ax_rich.set_xlim(-MARGIN, N_SIDE + MARGIN)
    ax_rich.set_ylim(N_SIDE + MARGIN, -MARGIN)       # row 0 on top
    ax_rich.set_aspect("equal"); ax_rich.axis("off")
    ax_rich.set_title("Ricchezza mediana del quartiere", fontsize=CBAR_LABEL_SIZE+2, weight="bold", pad=2,
                      color=LEGEND_TEXT_COLOR)

    # --- colorbar: shorter than the grid, vertically centred, "alta"/"bassa" labels ---
    grid_side = PANEL * N_SIDE / (N_SIDE + 2 * MARGIN)          # grid size inside the panel
    cb_h = grid_side * CBAR_FRAC
    cb_x = COL_X[1] + grid_side / 2 + CBAR_GAP
    cb_y = ROW_Y[0] + PANEL / 2 - cb_h / 2
    cax = fig.add_axes([cb_x / FIG_W, cb_y / FIG_H, CBAR_W / FIG_W, cb_h / FIG_H])
    cbar = fig.colorbar(mesh, cax=cax, orientation="vertical")
    cbar.outline.set_visible(False)
    cbar.set_ticks([]); cbar.ax.minorticks_off()
    cax.text(0.5, 1.03, "alta",  transform=cax.transAxes, ha="center", va="bottom",
             fontsize=CBAR_LABEL_SIZE, color=LEGEND_TEXT_COLOR)
    cax.text(0.5, -0.03, "bassa", transform=cax.transAxes, ha="center", va="top",
             fontsize=CBAR_LABEL_SIZE, color=LEGEND_TEXT_COLOR)

    # --- step slider (progress bar), top-left, aligned with the left column ---
    s0, s1 = steps[0], steps[-1]
    sl_x0 = COL_X[0] - PANEL / 2 + 0.8                        # inches (room for "inizio")
    sl_y0 = FIG_H - SLIDER_TOP - SLIDER_H
    sax = fig.add_axes([sl_x0 / FIG_W, sl_y0 / FIG_H, SLIDER_W / FIG_W, SLIDER_H / FIG_H])
    sax.set_xlim(s0, s1); sax.set_ylim(-1, 1); sax.axis("off")
    sax.plot([s0, s1], [0, 0], lw=SLIDER_LW, color=SLIDER_TRACK,
             solid_capstyle="round", clip_on=False, zorder=1)                     # track
    slider_fill, = sax.plot([s0, s0], [0, 0], lw=SLIDER_LW, color=SLIDER_COLOR,
                            solid_capstyle="round", clip_on=False, zorder=2)     # filled part
    slider_knob = sax.scatter([s0], [0], s=SLIDER_KNOB, color="white",
                              edgecolor=SLIDER_COLOR, linewidth=2.5,
                              clip_on=False, zorder=3)                           # knob
    sax.annotate("inizio", xy=(s0, 0), xytext=(-9, -6), textcoords="offset points",
                 ha="right", va="center", fontsize=LEGEND_FONTSIZE, color=LEGEND_TEXT_COLOR)
    sax.annotate("fine", xy=(s1, 0), xytext=(9, -6), textcoords="offset points",
                 ha="left", va="center", fontsize=LEGEND_FONTSIZE, color=LEGEND_TEXT_COLOR)
    step_txt = sax.text(s0, 0.45, "", ha="center", va="bottom", fontsize=LEGEND_FONTSIZE,
                        color=SLIDER_COLOR, weight="bold", clip_on=False)        # follows the knob

    # --- graphic legend (bottom centre), drawn in an axes measured in inches ---
    LW, LH = 9, LEGEND_H                              # legend box size (inches)
    lax = fig.add_axes([(FIG_W - LW) / 2 / FIG_W, 0.05 / FIG_H, LW / FIG_W, LH / FIG_H])
    lax.set_xlim(0, LW); lax.set_ylim(0, LH); lax.axis("off")

    # circle with a double arrow underneath showing its width
    r, ccx, ccy = 0.2, 1.2, 0.5
    lax.add_patch(Circle((ccx, ccy), r, facecolor=LEGEND_COLOR, alpha=0.5, edgecolor="none"))
    lax.annotate("", xy=(ccx - r, 0.14), xytext=(ccx + r, 0.14),
                 arrowprops=dict(arrowstyle="<|-|>", color=LEGEND_COLOR, lw=1.3,
                                 mutation_scale=10, shrinkA=0, shrinkB=0))
    lax.text(ccx + r + 0.3, ccy - 0.1, "residenti nel quartiere", va="center",
             fontsize=LEGEND_FONTSIZE, color=LEGEND_TEXT_COLOR)

    # thick flow arrow with two small arrows pointing at its edges (= thickness)
    fx0, fx1, fy, half = 5.0, 6.4, 0.4, 0.06         # half = half thickness (inches)
    lax.add_patch(FancyArrowPatch((fx0, fy), (fx1, fy), arrowstyle="-|>",
                                  mutation_scale=28, linewidth=half * 2 * 72,
                                  color=LEGEND_COLOR, shrinkA=0, shrinkB=0))
    mx = fx0 + 0.35
    for y_from, y_to in [(fy + 0.32, fy + half), (fy - 0.32, fy - half)]:
        lax.annotate("", xy=(mx, y_to), xytext=(mx, y_from),
                     arrowprops=dict(arrowstyle="-|>", color=LEGEND_COLOR, lw=1.3,
                                     mutation_scale=10, shrinkA=0, shrinkB=0))
    lax.text(fx1 + 0.2, fy, "flusso", va="center",
             fontsize=LEGEND_FONTSIZE, color=LEGEND_TEXT_COLOR)

    def update(i):
        step = steps[i]
        mesh.set_array(np.ma.masked_invalid(grids[grid_row[step]]).ravel())
        for t, ax in ax_flows.items():
            draw_flows(ax, t, step)
        slider_fill.set_xdata([s0, step])
        slider_knob.set_offsets([[step, 0]])
        step_txt.set_x(step)
        step_txt.set_text(f"Step {step}")
        return []

    update(0)
    frames = list(range(len(steps))) + [len(steps) - 1] * HOLD_LAST
    anim = FuncAnimation(fig, update, frames=frames, interval=1000 / FPS)
    anim.save("simulazione_completa.gif", writer=PillowWriter(fps=FPS), dpi=80)
    plt.close(fig)